# VL02 - Text Processing

We'll use two different libraries, one that is good for education, and a more powerful one typically used in production. We'll contrast both through this lecture:

1. **nltk**: Primarily used for teaching, research, and exploring algorithms in NLP, offering a huge collection of corpora and modular functions. Requires more effort and individual function calls (like sent_tokenize and separate downloads) as it gives you a wide range of options for each task.

2. **spaCy**: Optimized for speed, efficiency, and production use, processing text much faster for real-world applications like chatbots or large-scale data analysis. Uses advanced pre-trained statistical models to process text in one go, automatically providing accurate sentence segmentation, Part-of-Speech tags, and Named Entity Recognition (NER).

## 1. Loading the libraries
In the repository, you will find the scripts to install the dependencies required for this lab under the `/scripts` folder. 
- If you are in a YourAI cluster node, make sure to run the `install_env.sh` script to download additional dependencies. 
- If you are in `google collabs`, open the terminal and load the commands in the `install_google_collabs.sh`.

For this specific notebook, we also need:

````bash
python -m nltk.downloader punkt_tab averaged_perceptron_tagger_eng
python -m spacy download de_core_news_sm
pip install compound_split emoji
````

In [15]:
import re
try:
    import nltk
    from nltk.tokenize import sent_tokenize
    nltk.download('punkt', quiet=True)
except Exception:
    nltk = None
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
except Exception:
    nlp = None

## 2. Enconding and Unicode Normalisation

### 2.1 Different encodings
Unicode characters that look identical can be encoded in different ways. For example, the character ü may appear as:
- a single precomposed code point ü (U+00FC), or
- a decomposed sequence u (U+0075) + COMBINING DIAERESIS (U+0308).


In [2]:
"Mädchen"== "Mädchen"

False

In [3]:
s1 = 'Mädchen'       # 'Mädchen' NFC
s2 = 'Ma\u0308dchen' # 'Mädchen' NFD composed differently

print(f's1 = {s1}, s2 = {s2}')

s1 = Mädchen, s2 = Mädchen


In [4]:
print(f'{s1} == {s2} ?', s1 == s2)

Mädchen == Mädchen ? False


We see above two forms normally used:
- **NFC** (Normalised form C). It means composition. If possible, Unicode combines a base letter and a diacritic into one single character.

- **NFD** (Normalised form D). It means decomposition. It does the opposite: it splits a character into base letter + combining mark.

**NFC is usually the default choice** for general text preprocessing. Later we see that NFKC(Normalised Form KC - compability composition) is a good alternative when we care about compatibility.

In [5]:
import unicodedata
s2 = unicodedata.normalize('NFC', s2)

print(f'{s1} == {s2} ?', s1 == s2)

Mädchen == Mädchen ? True


### 2.2 Encoding as an exploit

#### 2.2.1 Full-Width characters
Unicode includes "Full-Width" characters (used in East Asian typography), which look like standard ASCII characters but are treated as completely different characters by a computer.

| Keyword | ASCII character | Full-width character | Unicode difference |
|---:|:---:|:---:|:---|
| F | `F` (U+0046) | `Ｆ` | U+FF26 |
| R | `R` (U+0052) | `Ｒ` | U+FF32 |
| E | `E` (U+0045) | `Ｅ` | U+FF25 |


In [6]:
message_standard = "Claim your FREE prize now!"
message_exploit = "Claim your ＦＲＥＥ prize now!"

def check_if_spam (message):
    if( "FREE" in message): 
        print("SPAM:\t", message)
    else: 
        print("HAM:\t", message)
        
check_if_spam(message_standard)
check_if_spam(message_exploit)

# Can we solve it with our normalisation?
message_normalised = unicodedata.normalize('NFC', message_exploit)
check_if_spam(message_normalised)

SPAM:	 Claim your FREE prize now!
HAM:	 Claim your ＦＲＥＥ prize now!
HAM:	 Claim your ＦＲＥＥ prize now!


##### What do we do?

In this situation, you might want to normalise to **NFKC**. NFKC normalizes characters that are compatibility variants of others (these are characters in Unicode intended to be typographic or compatibility forms). Examples NFKC will change:
- full-width latin letters Ｆ → F
- ligatures ﬁ → fi
- compatibility symbols ㎏ → kg
- circled numbers ① → 1
- superscripts ² → 2

In [7]:
# Some examples to run
samples = {
    "fullwidth": "Ｈｅｌｌｏ １２３",       # fullwidth Latin + fullwidth digits
    "ligature": "office ﬁle",             # 'ﬁ' ligature inside a word
    "compat_kg": "重量: ㎏",               # U+338F SQUARE KG -> 'kg'
    "superscript": "x² + y³",             # superscripts -> digits
    "circled": "① ② ③",                  # circled numbers -> digits
    "angstrom_sign": "\u212B",            # ANGSTROM SIGN (compat) -> 'Å'
    "umlaut" : 'Ma\u0308dchen',
}

def show(s):
    print("ORIG     :", s, " ->", [f"U+{ord(ch):04X}" for ch in s])
    nfc = unicodedata.normalize("NFC", s)
    nfkc = unicodedata.normalize("NFKC", s)
    print("NFC      :", nfc, " ->", [f"U+{ord(ch):04X}" for ch in nfc])
    print("NFKC     :", nfkc, " ->", [f"U+{ord(ch):04X}" for ch in nfkc])
    print("-" * 40)

for name, s in samples.items():
    print("SAMPLE:", name)
    show(s)

SAMPLE: fullwidth
ORIG     : Ｈｅｌｌｏ １２３  -> ['U+FF28', 'U+FF45', 'U+FF4C', 'U+FF4C', 'U+FF4F', 'U+0020', 'U+FF11', 'U+FF12', 'U+FF13']
NFC      : Ｈｅｌｌｏ １２３  -> ['U+FF28', 'U+FF45', 'U+FF4C', 'U+FF4C', 'U+FF4F', 'U+0020', 'U+FF11', 'U+FF12', 'U+FF13']
NFKC     : Hello 123  -> ['U+0048', 'U+0065', 'U+006C', 'U+006C', 'U+006F', 'U+0020', 'U+0031', 'U+0032', 'U+0033']
----------------------------------------
SAMPLE: ligature
ORIG     : office ﬁle  -> ['U+006F', 'U+0066', 'U+0066', 'U+0069', 'U+0063', 'U+0065', 'U+0020', 'U+FB01', 'U+006C', 'U+0065']
NFC      : office ﬁle  -> ['U+006F', 'U+0066', 'U+0066', 'U+0069', 'U+0063', 'U+0065', 'U+0020', 'U+FB01', 'U+006C', 'U+0065']
NFKC     : office file  -> ['U+006F', 'U+0066', 'U+0066', 'U+0069', 'U+0063', 'U+0065', 'U+0020', 'U+0066', 'U+0069', 'U+006C', 'U+0065']
----------------------------------------
SAMPLE: compat_kg
ORIG     : 重量: ㎏  -> ['U+91CD', 'U+91CF', 'U+003A', 'U+0020', 'U+338F']
NFC      : 重量: ㎏  -> ['U+91CD', 'U+91CF', 'U+003A', '

##### Defending against encoding exploits
When encoding attacks might be needed, or if we want to normalise for better search and matching, we can use NFKC. 
Does the code below defend agaisnt the attack?


In [8]:
message_normalised = unicodedata.normalize('NFKC', message_exploit)
check_if_spam(message_normalised)

SPAM:	 Claim your FREE prize now!


#### 2.2.2 Zero-width exploits
Compatibility normalization (`NFKC`) is very useful (full-width → ASCII, ligatures → constituent letters, etc.), but it does **not** remove invisible / zero-width characters such as ZWSP, ZWNJ, ZWJ or the BOM.  
These characters can appear accidentally (copy/paste, editors) and they will break literal substring/regex rules unless you remove or canonicalize them.

In [9]:
message = "Claim your F‌REE prize now!"
check_if_spam(message)

message_normalised = unicodedata.normalize('NFKC', message)
check_if_spam(message_normalised)

HAM:	 Claim your F‌REE prize now!
HAM:	 Claim your F‌REE prize now!


In [10]:
message

'Claim your F\u200cREE prize now!'

We can see above that we sneaked in an invisible character. We can deal with them by simply removing them. Characters and their unicode codes:

```
  ZWSP    ZWNJ     ZWJ    BOM   
 \u200B  \u200C  \u200D  \uFEFF   
```

In [11]:
_zero_re = re.compile("[\u200B\u200C\u200D\uFEFF]")

message_clean = _zero_re.sub('', message)
check_if_spam(message_clean)

SPAM:	 Claim your FREE prize now!


## 3. Sentence segmentation

### 3.1. Using regular expressions
We can use regular expressions to split a text into sentences.


In [12]:
text_short = "You have won a prize! Claim your gift now."

pattern = r'(?<=[.!?])\s+(?=[A-Z0-9"“\'\(\[])'   # split after .!? when next char looks like sentence start

sentences_short = re.split(pattern, text_short)

# Print sentences
def print_sentences (sentences):
    for i,s in enumerate(sentences,1):
        print(i, repr(s))

print_sentences(sentences_short)

1 'You have won a prize!'
2 'Claim your gift now.'


In [13]:
text_long = (
    "Congratulations! You have won $1,000.00. Contact Dr. O'Neil at 9:00 a.m. to claim your prize. "
    "Offer valid for U.S. residents only. See Sec. 3.2 for terms... Don't miss out! Visit www.example.com/free-offer "
    "or call 1-800-555-0199. Mr. Smith, CEO of Acme Ltd., says, \"Act now!\""
)

sentences = re.split(pattern, text_long)
print_sentences(sentences)

1 'Congratulations!'
2 'You have won $1,000.00.'
3 'Contact Dr.'
4 "O'Neil at 9:00 a.m. to claim your prize."
5 'Offer valid for U.S. residents only.'
6 'See Sec.'
7 '3.2 for terms...'
8 "Don't miss out!"
9 'Visit www.example.com/free-offer or call 1-800-555-0199.'
10 'Mr.'
11 'Smith, CEO of Acme Ltd., says, "Act now!"'


### 3.2 Using `ntlk` sentence segmentation
NLTK's `sent_tokenize` uses the Punkt sentence tokenizer — a data-driven model that detects sentence boundaries.


In [17]:
sentences_long = sent_tokenize(text_long, language='english')
print_sentences(sentences_long)

1 'Congratulations!'
2 'You have won $1,000.00.'
3 "Contact Dr. O'Neil at 9:00 a.m. to claim your prize."
4 'Offer valid for U.S. residents only.'
5 'See Sec.'
6 "3.2 for terms... Don't miss out!"
7 'Visit www.example.com/free-offer or call 1-800-555-0199.'
8 'Mr. Smith, CEO of Acme Ltd., says, "Act now!"'


### 3.3 Using `spacy` pipeline
spaCy performs sentence segmentation inside the pipeline via sentencizer/parser component. You access the results via `doc.sents`.

In [18]:
doc = nlp(text_long)

sentences_long_spacy = []

# Get sentences from the Doc object
# Note: can be also be done in python in one line!
#        sentences_long_spacy = [sent.text.strip() for sent in doc.sents]
for sent in doc.sents:
    sentence_text = sent.text.strip()
    sentences_long_spacy.append(sentence_text)

print_sentences(sentences_long_spacy)

1 'Congratulations!'
2 'You have won $1,000.00.'
3 "Contact Dr. O'Neil at 9:00 a.m. to claim your prize."
4 'Offer valid for U.S. residents only.'
5 "See Sec. 3.2 for terms... Don't miss out!"
6 'Visit www.example.com/free-offer or call 1-800-555-0199.'
7 'Mr. Smith, CEO of Acme Ltd., says, "Act now!"'


## 4. Tokenization

### 4.1 Using regular expressions

In [19]:
sent1 = 'You have won a prize!'
sent2 = "Don't miss this opportunity"

# word boundary -> word + optional ' or -  -> word boundary
re_tok = re.compile(r"\b\w[\w'\-]*\b")

print ( re_tok.findall(sent1) )
print ( re_tok.findall(sent2) )

['You', 'have', 'won', 'a', 'prize']
["Don't", 'miss', 'this', 'opportunity']


### 4.2 Using `nltk`
NLTK’s tokenizer is mainly rule-based, with some learned patterns.

In [20]:
from nltk.tokenize import word_tokenize
print ( word_tokenize(sent1) )
print ( word_tokenize(sent2) )

['You', 'have', 'won', 'a', 'prize', '!']
['Do', "n't", 'miss', 'this', 'opportunity']


### 4.3 Using `spacy`
The doc (output of `nlp(text)`) acts as a sequence of token objects, and you iterate on it to have access to all the tokens from a document. 

In [21]:
doc1 = nlp(sent1)
doc2 = nlp(sent2)
print ( [t.text for t in doc1] ) ## all tokens
print ( [t.text for t in doc2] )

['You', 'have', 'won', 'a', 'prize', '!']
['Do', "n't", 'miss', 'this', 'opportunity']


Tokens are annotated with different types of information, including whether they mark the start of a sentence. Indeed, to work with tokens at the level of **sentence** you can use `doc.sents`, which group tokens into sentence spans.

In [22]:
doc = nlp("You have won a prize! Claim your gift now.")

# Tokens from the document
print("===Tokens of the full document===")
for token in doc:
    print("   ", "is_start:", token.is_sent_start, "\t", token.text)

print("\n===Tokens by sentence===")
# Iterate over sentences (spans of tokens)
for i, sent in enumerate(doc.sents, 1):
    print(f"Sentence {i}: {sent.text}")
    
    # Iterate over tokens inside the sentence
    for token in sent:
        print("   ", "is_start:", token.is_sent_start, "\t", token.text)
        

===Tokens of the full document===
    is_start: True 	 You
    is_start: False 	 have
    is_start: False 	 won
    is_start: False 	 a
    is_start: False 	 prize
    is_start: False 	 !
    is_start: True 	 Claim
    is_start: False 	 your
    is_start: False 	 gift
    is_start: False 	 now
    is_start: False 	 .

===Tokens by sentence===
Sentence 1: You have won a prize!
    is_start: True 	 You
    is_start: False 	 have
    is_start: False 	 won
    is_start: False 	 a
    is_start: False 	 prize
    is_start: False 	 !
Sentence 2: Claim your gift now.
    is_start: True 	 Claim
    is_start: False 	 your
    is_start: False 	 gift
    is_start: False 	 now
    is_start: False 	 .


### 4.4 Challenges with German

In [23]:
# The Challenge: A standard tokenizer (like a simple regex or a basic word splitter)
# will treat the entire compound word as one token.

sent_german = "Der Krankenhaushaftpflichtversicherungsvertrag ist unklar."

# Simulate simple tokenization (e.g., splitting by space)
re.findall(r"\b\w[\w'\-]*\b", sent_german)

# Output (Error):
# ['Der', 'Krankenhaushaftpflichtversicherungsvertrag', 'ist', 'unklar.']

# The 'Solution' requires specialized tools (like spaCy's morphology component) 
# to split the word internally for proper analysis.

# Expected Correct Tokens for the compound word:
# ["Krankenhaus", "Haftpflicht", "Versicherung", "Vertrag"]

['Der', 'Krankenhaushaftpflichtversicherungsvertrag', 'ist', 'unklar']

Run `python -m spacy download de_core_news_sm` before 

In [24]:
nlp_de = spacy.load("de_core_news_sm")

# 2. Process the text
doc_de = nlp_de(sent_german)

# 3. Extract tokens
[token.text for token in doc_de]

['Der', 'Krankenhaushaftpflichtversicherungsvertrag', 'ist', 'unklar', '.']

Tokenization is separate from morphological analysis. Long German compounds (e.g. Krankenhaushaftpflichtversicherungsvertrag) are tokenized by spaCy as a single token — splitting them into meaningful parts (decompounding) requires extra processing.
For a fast, ready-to-run approach you can use CharSplit (compound-split):

``pip install compound-split``

`compound-split` returns ranked binary split candidates for a compound. It uses statistics over character patterns learned from data (i.e., it guesses splits that look like real words.
- The top candidate is the model’s preferred split (left + right).
- The output includes a numeric score; higher = more confident.
- Scores can be negative for rare/awkward splits — treat them as less confident.

You can recursively apply the splitter to split multi-part compounds into smaller parts.

In [25]:
from compound_split import char_split   # module exported by package

words = [
    "Krankenhaushaftpflichtversicherungsvertrag",  # hospital liability insurance contract
    "Krankenversicherung",                         # health insurance
    "Autobahnraststätte",                          # highway service area
    "Kindergartenfreundschaft",                    # kindergarten friendship
]

for w in words:
    splits = char_split.split_compound(w)   # returns ranked candidate (binary) splits
    print("WORD:", w)
    for score, left, right in splits[:3]:   # show top 3 candidates
        print(f"  score={score:.3f} -> {left} + {right}")
    print()

WORD: Krankenhaushaftpflichtversicherungsvertrag
  score=-0.135 -> Krankenhaushaftpflicht + Versicherungsvertrag
  score=-0.518 -> Krankenhaushaftpflichtversicherungs + Vertrag
  score=-0.644 -> Krankenhaus + Haftpflichtversicherungsvertrag

WORD: Krankenversicherung
  score=0.676 -> Kranken + Versicherung
  score=-0.708 -> Krankenver + Sicherung
  score=-0.952 -> Kranke + Nversicherung

WORD: Autobahnraststätte
  score=0.795 -> Autobahn + Raststätte
  score=-0.714 -> Auto + Bahnraststätte
  score=-1.113 -> Autobahnrast + Stätte

WORD: Kindergartenfreundschaft
  score=0.986 -> Kindergarten + Freundschaft
  score=-0.525 -> Kinder + Gartenfreundschaft
  score=-0.765 -> Kind + Ergartenfreundschaft



## 5. Case folding, punctuation, emoji handling

### 5.1 Case folding

**Case folding** is a general method to normalize text so that case differences are ignored. It is more comprehensive than simple lowercasing and follows Unicode rules for text comparison.

**Lowercasing** is a specific operation that converts uppercase letters into lowercase, but does not handle all language-specific cases.

In [26]:
tokens = ['WIN', 'Win', 'win']
[t.lower() for t in tokens]

['win', 'win', 'win']

In [27]:
tokens_de = ['Straße', 'STRASSE', "Osnabrück", "OSNABRUECK"]
t_lowered = [t.lower() for t in tokens_de]
t_folded = [t.casefold() for t in tokens_de]

print(t_lowered)
print(t_folded)

['straße', 'strasse', 'osnabrück', 'osnabrueck']
['strasse', 'strasse', 'osnabrück', 'osnabrueck']


**Diacritics**. Case folding handles case differences, but it does not remove accents or diacritics. In some tasks, we may also want to remove them. One way to do this is to first decompose characters into base letters and accent marks, then remove the accent marks.

In [29]:
def remove_diacritics(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) # decoposition (ü → u + ¨)
                   if not unicodedata.combining(ch)) # filters out accents
remove_diacritics("osnabrück")

'osnabruck'

### 5.2 Dealing with "special" characters
German (and many other European languages) use characters that are not ASCII — e.g. umlauts (ä/ö/ü), the sharp-S (ß) and other diacritics (é, ñ, …). How you handle them depends on the task. Below are the usual options and a short guideline.

- **Keep them (do nothing)**. Keep the original characters. This preserves all linguistic information and is recommended for most modern ML models and linguistic analyses.
- **Casefold only (caseless matching)**. Use Unicode case-folding (.casefold()) to compare case-insensitively. Important: .casefold() also maps ß → ss, unlike .lower().
- **Strip diacritics / ASCII transliteration**. Convert accented letters to closest ASCII equivalents (e.g., ü → u) using Unicode decomposition or a library (Unidecode, ICU). This loses diacritic information but can increase recall in legacy systems or ASCII-only contexts.
- **Orthographic mapping (German-style)**. Apply a language-aware mapping such as ä → ae, ö → oe, ü → ue, ß → ss. This preserves more of the original spelling conventions (useful for legacy matching, usernames, domain names, or keyboard variants).

What we choose depends on the type of task. When retreival tasks (matching, search) we probably want to strip it, for ML we probably don't want to strip  them, as we benefit from nuance


In [35]:
text = "🎉 WIN a FREE vacation,... NOW!! 🏖️"

# Using regular expressions
# Remove punctuation by replacing it with spaces
text_clean = re.sub(r"[.!?]+", " ", text)

print("original: ", text)
print("clean   : ", text_clean)

# With spacy
#   Notice that you would typically just filter the tokens (e.g., token.pos_ == PUNCT)

print("\nInspecting token PoS, looking for punctuations")
doc = nlp(text)
for token in doc:
    print("   ", token.text, " -> " , token.pos_, )

original:  🎉 WIN a FREE vacation,... NOW!! 🏖️
clean   :  🎉 WIN a FREE vacation,  NOW  🏖️

Inspecting token PoS, looking for punctuations
    🎉  ->  PROPN
    WIN  ->  PROPN
    a  ->  DET
    FREE  ->  ADJ
    vacation  ->  NOUN
    ,  ->  PUNCT
    ...  ->  PUNCT
    NOW  ->  ADV
    !  ->  PUNCT
    !  ->  PUNCT
    🏖  ->  NOUN
    ️  ->  X


Emojis are still there. We can strip them with the emoji library. First run

``pip install emoji``

In [37]:
import emoji
text = emoji.replace_emoji(text, replace=' <EMOJI> ')
print(text)

 <EMOJI>  WIN a FREE vacation,... NOW!!  <EMOJI> 


### 5.3 Stopwords
Stopwords are very common words (e.g., 'the', 'and', 'is') that often carry little meaning for a task. They are sometimes removed to reduce noise and focus on more informative words. NLTK provides predefined lists of stopwords for multiple languages.

In [38]:
try:
    nltk.data.find('corpora/stopwords')
except Exception:
    nltk.download('stopwords')
from nltk.corpus import stopwords

stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

#### Stopwords in NLTK

In [40]:
STOPWORDS = set(stopwords.words('english'))

# nltk
tokens = word_tokenize(text)
filtered_tokens = [ t for t in tokens if t not in STOPWORDS] 

print (tokens)
print(filtered_tokens)


['<', 'EMOJI', '>', 'WIN', 'a', 'FREE', 'vacation', ',', '...', 'NOW', '!', '!', '<', 'EMOJI', '>']
['<', 'EMOJI', '>', 'WIN', 'FREE', 'vacation', ',', '...', 'NOW', '!', '!', '<', 'EMOJI', '>']


#### Stopwords in spaCy

In spaCy, each token has an `is_stop` attribute that indicates whether it is a stopword, which can be used to filter them out.

In [ ]:
doc = nlp(text)
    
filtered_tokens_spacy = [token.text for token in doc 
                         if not token.is_stop and token.is_alpha]

print(filtered_tokens_spacy)

## 6. Stemming and Lematization 

### 6.1 Stemming
Stemming reduces words to a base or root form by removing suffixes (e.g.,'studying' → 'studi'). It is a simple, rule-based method and may produce forms that are not real words.

Common stemmers include the **Porter stemmer** (English) and the **Snowball stemmer** (multi-language), which we demonstrate below.

In [41]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball_en = SnowballStemmer("english")
snowball_de = SnowballStemmer("german")

print (porter.stem('study') )
print (snowball_en.stem('study') )
print (snowball_de.stem('Krankenhäuser') ) 

studi
studi
krankenhaus


#### Comparing stemmers
We compare different stemmers on English and German words to see how they reduce words to their root forms. This also highlights differences across languages and algorithms.

In [42]:
words_en = ['study', 'studies', 'studying', 'studied', 'run', 'running', 'ran', 'better', 'was', 'educate']
words_de = ['gehen', 'ging', 'gegangen', 'Krankenhaus', 'Krankenhäuser', 'arbeiten', 'arbeitete']

def compare_stems(words, lang='en'):
    print ("Language : ", lang)
    for w in words:
        p = porter.stem(w) 
        s_en = snowball_en.stem(w) 
        s_de = snowball_de.stem(w) 
        
        print(f" [{lang}] {w:15} | porter: {p:10} | snow_en: {s_en:10} | snow_de: {s_de:10}")
        
compare_stems(words_en, "en")  
compare_stems(words_de, "de")  

Language :  en
 [en] study           | porter: studi      | snow_en: studi      | snow_de: study     
 [en] studies         | porter: studi      | snow_en: studi      | snow_de: studi     
 [en] studying        | porter: studi      | snow_en: studi      | snow_de: studying  
 [en] studied         | porter: studi      | snow_en: studi      | snow_de: studied   
 [en] run             | porter: run        | snow_en: run        | snow_de: run       
 [en] running         | porter: run        | snow_en: run        | snow_de: running   
 [en] ran             | porter: ran        | snow_en: ran        | snow_de: ran       
 [en] better          | porter: better     | snow_en: better     | snow_de: bett      
 [en] was             | porter: wa         | snow_en: was        | snow_de: was       
 [en] educate         | porter: educ       | snow_en: educ       | snow_de: educat    
Language :  de
 [de] gehen           | porter: gehen      | snow_en: gehen      | snow_de: geh       
 [de] ging   

### 6.2 Lemmatization

#### Using ntlk
Lemmatization with NLTK requires some preparation, as the lemmatizer depends on the word's part of speech (POS), as seen in `wnl.lemmatize(token, pos)`.

Steps:

1. Tokenize the sentence into words
2. Assign POS tags to each word (`pos_tag`)
3. Map POS tags to WordNet categories (as `wnl.lemmatize(token, pos)` expects specific POS formats)
4. Lemmatize each word using its POS

In [45]:
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk import pos_tag

wnl = WordNetLemmatizer()

def treebank_to_wordnet_pos(treebank_tag):
    """Map NLTK/Treebank POS tags to WordNet POS tags for lemmatizer."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # sensible default

def lemmatize_sentence(sent: str):
    tokens = word_tokenize(sent)          # split into words
    tags = pos_tag(tokens)                # assign POS tags
    lemmas = []
    for token, tag in tags:
        wn_pos = treebank_to_wordnet_pos(tag)  # map to WordNet POS
        lemma = wnl.lemmatize(token, pos=wn_pos)  # get base form
        lemmas.append(lemma)
    return lemmas

# Example
print(lemmatize_sentence("You deserve better rewards for your loyalty."))

['You', 'deserve', 'good', 'reward', 'for', 'your', 'loyalty', '.']


#### Using spacy
The lemmatizer is already in spacy's pipeline, along with the POS. We can simply access the lemma with `token.lemma_`.

In [47]:
# To facilitate comparision, let's take the same words we used before
doc1 = nlp ("You deserve better rewards for your loyalty")

print ([token.lemma_ for token in doc1])

doc2 = nlp ("Get good benefits now - exclusively for loyal members!")

print ([token.lemma_ for token in doc2])

['you', 'deserve', 'well', 'reward', 'for', 'your', 'loyalty']
['get', 'good', 'benefit', 'now', '-', 'exclusively', 'for', 'loyal', 'member', '!']


#### Inspect the POS and lemma
You can insect the POS and the lemma below:

In [48]:
for token in doc1:
    print("{:<10} {:<10} {:<10}".format(
        token.text,             
        token.pos_,      # Coarse-grained POS tag (e.g., NOUN, VERB)
        token.lemma_,   
    ))

You        PRON       you       
deserve    VERB       deserve   
better     ADJ        well      
rewards    NOUN       reward    
for        ADP        for       
your       PRON       your      
loyalty    NOUN       loyalty   


In [49]:
from spacy import displacy
from IPython.display import HTML, display # Manually import from the standard location
# Get the raw HTML string

def display_parse_tree(doc):
    html_code = displacy.render(doc, style='dep', jupyter=False)
    
    # Use the standard IPython display to show the HTML
    display(HTML(html_code))
    
display_parse_tree(doc1)

## 7. A pre-processing pipeline

### 7.1 Removing punctuation before segementing sentences and tokens
We typically perform first segmentation and tokenization before removing punctuation. Otherwise this could affect the segmentation tasks.

In [52]:
def remove_punctuation (text):
    return re.sub(r"[\.,;:\"\\'(\)\[\]\\/\?@#\!\$%\^&\*_+=<>~`|]+", '', text) 

def tokenize (text):
    return word_tokenize(text)

text = "Don't click here!"

text_clean = remove_punctuation(text);
print ('over clean: ', tokenize(text_clean))
print ('over raw: ', tokenize(text))

over clean:  ['Dont', 'click', 'here']
over raw:  ['Do', "n't", 'click', 'here', '!']


### 7.2 Issues with Casefolding before lemmatisation
Lowercasing before POS tagging can change how words are interpreted. In some cases, this also affects the resulting lemma.

"Reading is a lovely town."
"They are reading the book." 

In [64]:
#text1 = "She is coming to US."
text1 = "May is a great month."

doc1 = nlp(text1.casefold())    
doc2 = nlp(text1)

display_parse_tree(doc1)
display_parse_tree(doc2)

# helper function
def get_lemma(doc):
    return [token.lemma_ for token in doc]

print("doc1", get_lemma(doc1))
print("doc2", get_lemma(doc2))


doc1 ['may', 'be', 'a', 'great', 'month', '.']
doc2 ['May', 'be', 'a', 'great', 'month', '.']


### 7.3 Issues with stopword removal before lemmatisation
Removing stopwords before lemmatization **can affect the structure of the sentence**, which may lead to less reliable POS tagging. We have already seen that even simple normalization steps (e.g., case folding) can influence this.

In addition, stopword removal is form-based: if applied before normalization or lemmatization, some variants of stopwords (e.g., inflected forms or contractions) **may not match the stopword list** and remain in the text.

In [68]:
text = "The man wasn't running to the store"
tokens = word_tokenize(text)

stoplist = stopwords.words('english')

print("Original tokens:", tokens)

# (A) Remove stopwords BEFORE lemmatization
removed_before = [t for t in tokens if t.lower() not in stoplist]
print("Before lemmatization:", removed_before)

# (B) Lemmatize first, THEN remove stopwords
doc = nlp(text)
lemmas = [tok.lemma_ for tok in doc]
filtered_after = [tok for tok in lemmas if tok not in stoplist]

print("After lemmatization:", filtered_after)
print("Lemmas:", lemmas)

Original tokens: ['The', 'man', 'was', "n't", 'running', 'to', 'the', 'store']
Before lemmatization: ['man', "n't", 'running', 'store']
After lemmatization: ['man', 'run', 'store']
Lemmas: ['the', 'man', 'be', 'not', 'run', 'to', 'the', 'store']


## 8. Practical notes about `spacy`  pipeline
spaCy processes text through a pipeline of components (e.g., tokenizer, tagger, parser, NER). Each component adds annotations to the tokens.

In [69]:
print(nlp.pipeline)
print(nlp.pipe_names)

[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec object at 0x7f2d11db5f00>), ('tagger', <spacy.pipeline.tagger.Tagger object at 0x7f2d11db5d80>), ('parser', <spacy.pipeline.dep_parser.DependencyParser object at 0x7f2d11ec2f10>), ('attribute_ruler', <spacy.pipeline.attributeruler.AttributeRuler object at 0x7f2d11b5b580>), ('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer object at 0x7f2d11b2c680>), ('ner', <spacy.pipeline.ner.EntityRecognizer object at 0x7f2d11ec3060>)]
['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


You can control which components of the spaCy pipeline are used, depending on your needs. If you only require tokenization, you can use the tokenizer directly (`nlp.make_doc`) or disable unnecessary components to improve efficiency.

In [71]:
# NER
text = "Maria was walking in Paris. That was far from the United States of America"
doc = nlp(text)
print([(ent.text, ent.label_) for ent in doc.ents])

[('Maria', 'PERSON'), ('Paris', 'GPE'), ('the United States of America', 'GPE')]
Tokens (make_doc): ['Maria', 'was', 'walking', 'in', 'Paris', '.', 'That', 'was', 'far', 'from', 'the', 'United', 'States', 'of', 'America']
Lema  (make_doc): ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Tokens (make_doc): ['Maria', 'was', 'walking', 'in', 'Paris', '.', 'That', 'was', 'far', 'from', 'the', 'United', 'States', 'of', 'America']


In [ ]:
# Running just the tokenizer
doc_tokens_only = nlp.make_doc(text)
print("Tokens (make_doc):", [t.text for t in doc_tokens_only])
print("Lema  (make_doc):", [t.lemma_ for t in doc_tokens_only])

In [ ]:
# Disabling components
doc = nlp(text, disable=["ner", "parser"])
print("Tokens (make_doc):", [t.text for t in doc_tokens_only])

## References

https://spacy.io/usage/processing-pipelines